# EACF ridge morphology

This tutorial keeps the complete frequency–lag map instead of reducing it immediately to one peak. The plots show how an oscillation ridge changes as the band-pass filter is moved through trial centre frequencies.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from urdr import (
    CoherentSignalConfig,
    SegmentSystematicConfig,
    SimulationConfig,
    benchmark_morphology_veto,
    compute_eacf,
    compute_eacf_map,
    eacf_morphology,
    make_observing_window,
    simulate_time_series,
)

## Simulate a gapped oscillator

Missing cadences remain on the regular grid and are carried by the observing mask. The same mask is used for signal and calibration simulations.

In [ ]:
window = make_observing_window(
    duration_days=1.0,
    cadence_seconds=120.0,
    gaps_days=((0.49, 0.51),),
)
simulation = SimulationConfig(
    white_noise_sigma=0.2,
    granulation_amplitude=0.1,
    numax_uhz=1000.0,
    delta_nu_uhz=100.0,
    envelope_width_uhz=400.0,
    oscillation_amplitude=1.5,
)
centres = np.linspace(700.0, 1300.0, 9)
series = simulate_time_series(
    window, simulation, np.random.default_rng(8), include_oscillations=True
)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(series.time[series.observed], series.flux[series.observed], lw=0.7)
ax.axvspan(0.49, 0.51, color="0.85", label="gap")
ax.set(xlabel="Time [days]", ylabel="Flux", title="Simulated input time series")
ax.legend()
plt.show()

## ACFs at individual test frequencies

Each curve is the squared, normalised ACF of a time series filtered around one trial frequency. The vertical lines mark the expected first and second seismic lags, (1/\Delta\nu) and (2/\Delta\nu). A filter centred on the oscillation envelope should reveal the strongest repeated structure.

In [ ]:
test_centres = [750.0, 1000.0, 1250.0]
expected_lag = 1e6 / simulation.delta_nu_uhz

fig, ax = plt.subplots(figsize=(9, 4))
for centre in test_centres:
    lags, values = compute_eacf(
        series,
        centre_frequency_uhz=centre,
        filter_width_uhz=500.0,
        max_lag_seconds=25_000.0,
    )
    ax.plot(lags / 3600.0, values, label=f"{centre:.0f} µHz")
for multiple in (1, 2):
    ax.axvline(
        multiple * expected_lag / 3600.0,
        color="k",
        ls="--",
        alpha=0.55,
        label="expected seismic lags" if multiple == 1 else None,
    )
ax.set(xlabel="Lag [hours]", ylabel="Normalised ACF²", title="Filtered ACF slices")
ax.legend(ncols=2)
plt.show()

## Frequency–lag ACF map

Stacking many trial centres forms the EACF map. A genuine oscillation envelope should produce a localised ridge near (
u_{\max}), supported near both expected lags.

In [ ]:
eacf_map = compute_eacf_map(
    series, centres, 500.0, max_lag_seconds=25_000.0
)

fig, ax = plt.subplots(figsize=(9, 5))
mesh = ax.pcolormesh(
    eacf_map.lags_seconds / 3600.0,
    eacf_map.centre_frequencies_uhz,
    eacf_map.values,
    shading="auto",
    cmap="magma",
)
for multiple in (1, 2):
    ax.axvline(multiple * expected_lag / 3600.0, color="cyan", ls="--", lw=1)
ax.axhline(simulation.numax_uhz, color="white", ls=":", lw=1)
ax.set(
    xlim=(1.5, 7.0),
    xlabel="Lag [hours]",
    ylabel="Filter centre [µHz]",
    title="Frequency–lag EACF map",
)
fig.colorbar(mesh, ax=ax, label="Normalised ACF²")
plt.show()

In [ ]:
diagnostics = eacf_morphology(
    eacf_map,
    delta_nu_uhz=simulation.delta_nu_uhz,
    expected_numax_uhz=simulation.numax_uhz,
    envelope_width_uhz=simulation.envelope_width_uhz,
)
diagnostics

The morphology numbers describe the visible structure: peak location, connected width and fill, contrast, roughness, and support at the second ACF peak. Their scale depends on the target and observing window, so Urdr calibrates them using exact-window injections rather than applying universal cuts.

In [ ]:
benchmark = benchmark_morphology_veto(
    window=window,
    simulation=simulation,
    centre_frequencies_uhz=centres,
    filter_width_uhz=500.0,
    delta_nu_grid_uhz=np.array([90.0, 100.0, 110.0]),
    coherent_contaminants={
        "single_line": CoherentSignalConfig(1000.0, 0.8),
    },
    segment_systematics={
        "variance_jump": [
            SegmentSystematicConfig(0.5, 1.0, amplitude_scale=4.0)
        ],
    },
    realizations=8,
    target_false_positive_rate=0.25,
    target_signal_retention=0.9,
    max_lag_seconds=25_000.0,
    seed=42,
)
{
    "signal_before": benchmark.raw_signal_detection_rate,
    "signal_after": benchmark.accepted_signal_detection_rate,
    "contaminants_before": benchmark.raw_contaminant_detection_rate,
    "contaminants_after": benchmark.accepted_contaminant_detection_rate,
}